# Medical Triage Agent AI POC
## Kaggle Notebooks Training Pipeline


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Vérifications
print("GPU :", torch.cuda.get_device_name(0))   # doit afficher Tesla T4 ou P100 selon le tier Kaggle sélectionné
print("VRAM totale :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("BF16 natif :", torch.cuda.is_bf16_supported())   # → False sur T4/P100 ✓

In [ ]:
from __future__ import annotations
import logging
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger('training_pipeline')


# Section 2 — Bootstrap Runtime


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import importlib.metadata

def ensure_package(module_name, pip_name=None, min_version=None):
    spec = importlib.util.find_spec(module_name)
    if spec is not None:
        if min_version is None:
            return  # Installé, pas de contrainte de version → OK
        try:
            installed = importlib.metadata.version(module_name)
            from packaging.version import Version
            if Version(installed) >= Version(min_version):
                return  # Version suffisante → OK
            print(f"⚠️ {module_name} {installed} < {min_version} requis → mise à jour...")
        except Exception:
            return
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-U', pip_name or module_name],
        check=True
    )

In [ ]:
import importlib.metadata

DEPENDENCIES = [
    ('torch',            None,                None),
    ('transformers',     None,                None),
    ('datasets',         None,                None),
    ('accelerate',       None,                None),
    ('trl',              None,                None),
    ('peft',             None,                None),
    ('bitsandbytes',     None,                None),
    ('unsloth',          None,                None),
    ('huggingface_hub',  'huggingface_hub',   None),
    ('wandb',            None,                None),
    ('evaluate',         None,                None),
    ('sentencepiece',    None,                None),
    ('safetensors',      None,                None),
    ('torchao',          None,                '0.16.0'),  # ← version minimale
]

print("📦 Vérification des dépendances :\n")
for m, p, v in DEPENDENCIES:
    ensure_package(m, p, v)
    try:
        version = importlib.metadata.version(m)
        print(f"  ✅ {m:<20} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  ❌ {m:<20} non trouvé")

print("\n✅ Toutes les dépendances sont prêtes.")

In [ ]:
# !pip install -U unsloth

In [ ]:
# NOTE (spécifique Kaggle) : nécessite que l'option "Internet" soit activée
# pour ce Notebook (Settings > Internet > On), désactivée par défaut sur Kaggle.
!git clone https://github.com/raym648/medical-triage-agent-ai-poc.git

In [ ]:
%cd /kaggle/working/medical-triage-agent-ai-poc

In [ ]:
!ls

In [ ]:
PROJECT_ROOT=Path.cwd().resolve()
while PROJECT_ROOT!=PROJECT_ROOT.parent:
    if (PROJECT_ROOT/'backend').exists() and (PROJECT_ROOT/'frontend').exists():
        break
    PROJECT_ROOT=PROJECT_ROOT.parent
else:
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
required=['backend/app/training/kaggle','backend/app/training/sft','backend/app/training/dpo','backend/app/training/evaluation']
missing=[p for p in required if not (PROJECT_ROOT/Path(p)).exists()]
if missing:
    raise FileNotFoundError(missing)


# Section 3


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['HF_TOKEN_06'] = user_secrets.get_secret('HF_TOKEN_06')  # ← ligne ajoutée

# NOTE (spécifique Kaggle) : pas d'équivalent Google Drive — la persistance
# se limite à /kaggle/working pour la durée de la session (ou à un commit
# de Version pour conserver les sorties au-delà de la session).

from backend.app.training.kaggle.kaggle_environment import log_environment_info
from backend.app.training.kaggle.kaggle_gpu_detector import log_gpu_info
from backend.app.training.kaggle.kaggle_hf_login import ensure_hf_login
from backend.app.training.kaggle.kaggle_checkpoint_sync import create_default_checkpoint_sync

environment = log_environment_info()
gpu_info = log_gpu_info()
hf_status = ensure_hf_login()
checkpoint_sync = create_default_checkpoint_sync(output_dir='/kaggle/working/outputs')
checkpoint_sync.get_status()

print("✅ HF Login :", hf_status)


# Section 4


In [ ]:
# À exécuter AVANT la cellule DEPENDENCIES
# !pip install -q "torchao>=0.16.0"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.sft.train_sft',run_name='__main__')


In [ ]:
import trl, inspect
print(trl.__version__)
print(list(inspect.signature(trl.DPOConfig.__init__).parameters.keys()))

In [ ]:
import trl
import transformers

print("TRL :", trl.__version__)
print("Transformers :", transformers.__version__)

In [ ]:
from trl import DPOTrainer
import inspect

print(inspect.signature(DPOTrainer.__init__))

In [ ]:
import torch
import transformers
import trl
import peft
import bitsandbytes

print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("trl           :", trl.__version__)
print("peft          :", peft.__version__)
print("bitsandbytes  :", bitsandbytes.__version__)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.dpo.train_dpo',run_name='__main__')


# Section 6


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.evaluation.clinical_eval_runner',run_name='__main__')


# Section 7


In [ ]:
from backend.app.training.kaggle.kaggle_checkpoint_sync import create_default_checkpoint_sync
checkpoint_sync=create_default_checkpoint_sync(output_dir='/kaggle/working/outputs')
print(checkpoint_sync.sync_latest_checkpoint())


# Section 8


In [ ]:
from backend.app.training.kaggle.kaggle_checkpoint_sync import create_default_checkpoint_sync
checkpoint_sync=create_default_checkpoint_sync(output_dir='/kaggle/working/outputs')
print(checkpoint_sync.get_status())
